Derin Öğrenme ile Görüntü İşleme: CNN, U-Net ve Mask R-CNN

Bu not defteri, görüntü işlemede en çok kullanılan üç derin öğrenme mimarisini kavramsal açıklama + satır satır yorumlanmış kod formatında ele alır:

CNN (Convolutional Neural Network) — Görüntü sınıflandırma
U-Net — Piksel bazlı segmentasyon (görüntüdeki her pikseli sınıflandırma)
Mask R-CNN — Nesne tespiti + örnek (instance) segmentasyonu
Not: Bu defter eğitim amaçlıdır. Küçük/sentetik veri veya torchvision gibi hazır ağırlıklarla çalışacak şekilde tasarlanmıştır, böylece kod hiçbir ek veri seti indirmeden çalıştırılabilir ve mimarilerin nasıl kurulduğu görülebilir. Gerçek bir projede kendi veri setinizle eğitim (training) döngüsünü genişletmeniz gerekir.

Kullanılan kütüphane: PyTorch (torch, torchvision)


In [1]:
# Gerekli kütüphaneleri kontrol edip kuruyoruz
# torch: tensör işlemleri ve sinir ağı katmanları için temel kütüphane
# torchvision: hazır modeller (Mask R-CNN dahil) ve görüntü dönüşümleri için
!pip install torch torchvision --quiet


In [ ]:
import torch                     # Ana derin öğrenme kütüphanesi (tensörler, autograd)
import torch.nn as nn            # Sinir ağı katmanlarını içerir (Conv2d, Linear, vb.)
import torch.nn.functional as F  # Aktivasyon fonksiyonları gibi durumsuz (stateless) işlemler
import numpy as np                # Sayısal işlemler ve dizi (array) manipülasyonu için

torch.manual_seed(42)  # Tekrar üretilebilirlik için rastgelelik tohumunu (seed) sabitliyoruz
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU varsa kullan, yoksa CPU
print(f"Kullanılan cihaz: {device}")


Kullanılan cihaz: cpu


In [ ]:
class BasitCNN(nn.Module):
    """
    Basit bir görüntü sınıflandırma CNN'i.
    Giriş: (batch, 1, 28, 28) boyutunda gri tonlamalı görüntü
    Çıkış: (batch, 10) boyutunda sınıf skorları (logits)
    """
    def __init__(self, num_classes=10):
        super().__init__()  # nn.Module'ün kurucusunu çağırıyoruz (zorunlu)

        # --- 1. Konvolüsyon Bloğu ---
        # in_channels=1: gri tonlamalı görüntü (RGB olsaydı 3 olurdu)
        # out_channels=16: 16 farklı filtre öğrenilecek, yani 16 özellik haritası üretilecek
        # kernel_size=3: 3x3 boyutunda filtre
        # padding=1: kenarlarda bilgi kaybını önlemek için görüntü etrafına 1 piksel sıfır ekle
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)  # Eğitimi stabilize eden normalizasyon katmanı

        # --- 2. Konvolüsyon Bloğu ---
        # Bir önceki katmanın çıktısı (16 kanal) burada girdi olur, 32 kanala çıkarılır
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)

        # Havuzlama: her 2x2'lik alanın maksimumunu alarak boyutu yarıya indirir
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # İki kez pool uygulanacağı için: 28 -> 14 -> 7
        # Son özellik haritası boyutu: 32 kanal x 7 x 7
        self.fc1 = nn.Linear(32 * 7 * 7, 128)   # Tam bağlantılı katman: özellikleri 128 boyuta indirir
        self.dropout = nn.Dropout(0.3)          # Aşırı öğrenmeyi (overfitting) azaltmak için nöronların %30'unu rastgele söndürür
        self.fc2 = nn.Linear(128, num_classes)  # Son katman: her sınıf için bir skor üretir

    def forward(self, x):
        # x boyutu: (batch, 1, 28, 28)

        x = self.conv1(x)          # Konvolüsyon uygula -> (batch, 16, 28, 28)
        x = self.bn1(x)            # Batch normalization uygula
        x = F.relu(x)              # Negatif değerleri 0 yapan aktivasyon fonksiyonu
        x = self.pool(x)           # Boyutu yarıya indir -> (batch, 16, 14, 14)

        x = self.conv2(x)          # İkinci konvolüsyon -> (batch, 32, 14, 14)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool(x)           # -> (batch, 32, 7, 7)

        x = torch.flatten(x, 1)    # 2D özellik haritalarını 1D vektöre düzleştir -> (batch, 32*7*7)
        x = self.fc1(x)            # Tam bağlantılı katmandan geçir -> (batch, 128)
        x = F.relu(x)
        x = self.dropout(x)        # Dropout uygula (sadece eğitim sırasında aktif)
        x = self.fc2(x)            # Son katman -> (batch, num_classes) ham skorlar (logits)

        return x


In [ ]:
# Modeli oluşturup cihaza (CPU/GPU) taşıyoruz
model_cnn = BasitCNN(num_classes=10).to(device)
print(model_cnn)

# Sahte (dummy) bir görüntü batch'i ile ileri yayılımı (forward pass) test edelim
# 8 adet, 1 kanallı, 28x28 boyutunda rastgele görüntü
sahte_girdi = torch.randn(8, 1, 28, 28).to(device)
cikti = model_cnn(sahte_girdi)

print(f"Girdi boyutu : {sahte_girdi.shape}")
print(f"Çıktı boyutu : {cikti.shape}  # (batch_size=8, num_classes=10)")


BasitCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1568, out_features=128, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)
Girdi boyutu : torch.Size([8, 1, 28, 28])
Çıktı boyutu : torch.Size([8, 10])  # (batch_size=8, num_classes=10)


In [ ]:
# Kayıp fonksiyonu: çok sınıflı sınıflandırma için standart seçim
criterion = nn.CrossEntropyLoss()

# Optimizer: ağırlıkları gradyanlara göre güncelleyen algoritma
optimizer = torch.optim.Adam(model_cnn.parameters(), lr=1e-3)

# --- Tek bir eğitim adımı örneği (sahte veriyle) ---
sahte_etiketler = torch.randint(0, 10, (8,)).to(device)  # 8 örnek için rastgele doğru sınıflar

optimizer.zero_grad()                    # Önceki adımdan kalan gradyanları sıfırla
cikti = model_cnn(sahte_girdi)           # İleri yayılım: tahminleri hesapla
loss = criterion(cikti, sahte_etiketler) # Tahmin ile gerçek etiket arasındaki hatayı hesapla
loss.backward()                          # Geri yayılım: her ağırlık için gradyanı hesapla
optimizer.step()                         # Ağırlıkları gradyana göre güncelle

print(f"Örnek eğitim adımı kaybı (loss): {loss.item():.4f}")


Örnek eğitim adımı kaybı (loss): 2.4422


U-NET

In [ ]:
class ConvBlok(nn.Module):
    """
    U-Net'te tekrar tekrar kullanılan temel yapı taşı:
    (Konvolüsyon -> BatchNorm -> ReLU) x 2
    Bu blok hem encoder hem decoder tarafında kullanılır.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.blok = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),  # Uzamsal boyutu koru (padding=1)
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),  # İkinci konvolüsyon, kanal sayısı sabit kalır
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.blok(x)  # Sequential içindeki tüm katmanları sırayla uygula


In [ ]:
class UNet(nn.Module):
    """
    Basitleştirilmiş U-Net mimarisi.
    Giriş : (batch, in_ch, H, W)  -- örn. RGB görüntü (in_ch=3)
    Çıkış : (batch, num_classes, H, W) -- her piksel için sınıf skorları
    """
    def __init__(self, in_ch=3, num_classes=1, ozellikler=[64, 128, 256, 512]):
        super().__init__()

        self.encoder = nn.ModuleList()  # Daraltma yolundaki blokları tutacak liste
        self.decoder = nn.ModuleList()  # Genişletme yolundaki blokları tutacak liste
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)  # Her encoder adımından sonra boyutu yarıya indirir

        # --- ENCODER (Daraltma Yolu) oluşturuluyor ---
        kanal = in_ch
        for f in ozellikler:
            self.encoder.append(ConvBlok(kanal, f))  # örn: 3->64, 64->128, 128->256, 256->512
            kanal = f

        # --- BOTTLENECK (En dar/derin katman, encoder ile decoder arasındaki köprü) ---
        self.bottleneck = ConvBlok(ozellikler[-1], ozellikler[-1] * 2)  # 512 -> 1024

        # --- DECODER (Genişletme Yolu) oluşturuluyor ---
        # Ozellikler listesini ters çeviriyoruz çünkü decoder küçükten büyüğe değil,
        # derinden (dar) sığa (geniş) doğru ilerler
        for f in reversed(ozellikler):
            # Transpoze konvolüsyon: özellik haritasının boyutunu 2 katına çıkarır (upsampling)
            self.decoder.append(
                nn.ConvTranspose2d(f * 2, f, kernel_size=2, stride=2)
            )
            # Skip connection ile birleştirildikten sonra kanal sayısı 2 katına çıkar (f*2),
            # bu yüzden ConvBlok girişi f*2 olarak ayarlanır
            self.decoder.append(ConvBlok(f * 2, f))

        # Son katman: kanal sayısını istenen sınıf sayısına indirir (1x1 konvolüsyon)
        self.son_katman = nn.Conv2d(ozellikler[0], num_classes, kernel_size=1)

    def forward(self, x):
        skip_baglantilari = []  # Her encoder seviyesinin çıktısını burada saklayacağız

        # --- Encoder'dan geçiş ---
        for enc_blok in self.encoder:
            x = enc_blok(x)                # Konvolüsyon bloğunu uygula
            skip_baglantilari.append(x)    # Havuzlamadan ÖNCEKİ hali sakla (decoder'a taşınacak)
            x = self.pool(x)               # Boyutu yarıya indir

        # --- Bottleneck ---
        x = self.bottleneck(x)  # En derin katman

        # Skip bağlantılarını decoder'da kullanmak için ters çeviriyoruz
        # (en son eklenen encoder çıktısı, decoder'da en önce kullanılır)
        skip_baglantilari = skip_baglantilari[::-1]

        # --- Decoder'dan geçiş ---
        # self.decoder listesinde ikişerli gidiyoruz: [upsample, convblok, upsample, convblok, ...]
        for i in range(0, len(self.decoder), 2):
            x = self.decoder[i](x)                    # Transpoze konvolüsyon ile boyutu büyüt
            skip = skip_baglantilari[i // 2]           # İlgili encoder seviyesinin sakladığı özellik haritası

            # Boyutlar (yuvarlama hatalarından dolayı) tam eşleşmeyebilir, gerekirse kırp
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:])  # skip ile aynı H,W boyutuna getir

            x = torch.cat((skip, x), dim=1)           # KRİTİK ADIM: skip connection -> kanal boyutunda birleştir
            x = self.decoder[i + 1](x)                # ConvBlok ile işle

        return self.son_katman(x)  # Son 1x1 konvolüsyon ile sınıf haritasını üret
